# Mervin/Mervis -- Qwen2.5-7B LoRA fine-tune (Google Colab)

LoRA fine-tunes **Qwen2.5-7B-Instruct** (Apache 2.0) on the Mervin/Mervis persona
dataset with [Unsloth](https://github.com/unslothai/unsloth), exports a 4-bit
**Q4_K_M GGUF** (~4.7 GB), and uploads it to Hugging Face -- where `serve.py`
auto-downloads it.

This replaces the old MLX-only Qwen 3.5-4B: **Qwen2.5 is a standard transformer
that converts to GGUF cleanly**, so it runs through llama.cpp on CPU like every
other arena model -- no MLX, no Mac requirement.

**No Mervin system prompt.** Training data is bare `user -> assistant`; the
persona comes purely from fine-tuning. (Qwen2.5's chat template injects its own
generic "You are Qwen..." system line at both train and inference time -- that's
constant boilerplate, not a persona instruction, so behavior still comes from the
fine-tune.) Verified on A100: clean Mervin/Mervis from a user-only message.

### Before you run
- Runtime -> Change runtime type -> **GPU** (A100/L4; T4 works, slower).
- Add a Colab **secret** `HF_TOKEN` (key icon) with a Hugging Face **write** token.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
!pip install --upgrade -q unsloth unsloth_zoo
!pip install -q "transformers==4.56.2" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0"
!pip uninstall -q -y torchao
import transformers, trl, datasets
print("transformers", transformers.__version__, "| trl", trl.__version__, "| datasets", datasets.__version__)

In [ ]:
import torch
from unsloth import FastLanguageModel
MAX_SEQ_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen2.5-7B-Instruct",   # not gated
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    full_finetuning = False,
)
print("loaded Qwen2.5-7B-Instruct")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)
model.print_trainable_parameters()

## Dataset -- no Mervin system prompt

In [ ]:
import csv, io, urllib.request
from datasets import Dataset
CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"
raw = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
rows = list(csv.DictReader(io.StringIO(raw)))
print(f"Loaded {len(rows)} rows")
# Full user->assistant turn via Qwen's native template (it adds its own generic
# system line; we add no Mervin instruction). Persona is learned from the targets.
def to_text(r):
    return {"text": tokenizer.apply_chat_template(
        [{"role":"user","content":r["prompt"]},{"role":"assistant","content":r["response"]}],
        tokenize=False, add_generation_prompt=False)}
ds = Dataset.from_list([to_text(r) for r in rows])
print(ds[0]["text"][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        num_train_epochs=3, learning_rate=2e-4, warmup_ratio=0.1,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
        logging_steps=10, seed=3407, output_dir="outputs", report_to="none",
    ),
)
trainer.train()

## Sanity check -- bare user message

In [ ]:
FastLanguageModel.for_inference(model)
def ask(q):
    prompt = tokenizer.apply_chat_template([{"role":"user","content":q}], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inp, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
for q in ["What is 2+2?", "Tell me about Mondays.", "What is the capital of France?"]:
    print("Q:", q); print(ask(q)); print("-"*60)

## Export Q4_K_M GGUF + upload to Hugging Face

Uploads to `freeideas/merv-qwen2.5-7b` as `model-q4_k_m.gguf`. Token from the
Colab `HF_TOKEN` secret.

In [ ]:
import glob, os
model.save_pretrained_gguf("qwen-merv-gguf", tokenizer, quantization_method="q4_k_m")
src = glob.glob("/content/**/qwen2.5*Q4_K_M.gguf", recursive=True)[0]
print("GGUF:", round(os.path.getsize(src)/1e9, 2), "GB", src)

from google.colab import userdata
from huggingface_hub import HfApi
tok = userdata.get("HF_TOKEN"); repo = "freeideas/merv-qwen2.5-7b"
api = HfApi(); api.create_repo(repo, repo_type="model", exist_ok=True, token=tok)
api.upload_file(path_or_fileobj=src, path_in_repo="model-q4_k_m.gguf",
                repo_id=repo, repo_type="model", token=tok)
print("done -> https://huggingface.co/" + repo)

## In the arena

`serve.py` and `index.html` carry a `qwen` entry (now a llama.cpp GGUF, not MLX),
so any host running `serve.py` auto-downloads `model-q4_k_m.gguf` from
`freeideas/merv-qwen2.5-7b` on startup and it appears in the dropdown.